# Module 2.4 — Metadata Management

Metadata makes retrieval smarter. Every `Document` carries a `metadata` dict that can be used for **filtering, routing, and lineage tracking**.

Key metadata fields to add:
- `source` — file/URL path
- `page` — page number
- `chunk_id` — unique chunk identifier
- `author`, `created_at`, `version` — document lineage

In [ ]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from datetime import datetime
import hashlib

# ── Metadata-enriched document pipeline ──────────────────────────────────────
def enrich_metadata(docs: list[Document], extra: dict) -> list[Document]:
    """Add extra metadata fields to every document in the list."""
    enriched = []
    for i, doc in enumerate(docs):
        meta = {
            **doc.metadata,
            **extra,
            "chunk_id"   : hashlib.md5(doc.page_content.encode()).hexdigest()[:8],
            "chunk_index": i,
            "ingested_at": datetime.utcnow().isoformat(),
        }
        enriched.append(Document(page_content=doc.page_content, metadata=meta))
    return enriched


raw_text = """Chapter 1: Introduction to Machine Learning
Machine learning is a subset of artificial intelligence that enables systems to learn from data.

Chapter 2: Supervised Learning
In supervised learning, models are trained on labelled data to make predictions.

Chapter 3: Unsupervised Learning
Unsupervised learning discovers hidden patterns in data without labels.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
raw_docs = splitter.create_documents([raw_text], metadatas=[{"source": "ml_textbook.pdf"}])

enriched_docs = enrich_metadata(raw_docs, {
    "author"  : "Jane Doe",
    "version" : "v2.1",
    "category": "textbook",
})

for d in enriched_docs:
    print(f"chunk_id={d.metadata['chunk_id']} | index={d.metadata['chunk_index']}")
    print(f"  {d.page_content[:80].strip()}")
    print(f"  metadata → {d.metadata}\n")


In [ ]:
# ── Metadata filtering in retrieval ──────────────────────────────────────────
embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(enriched_docs, embeddings, collection_name="meta_demo")

# Filter: only return chunks from version v2.1 authored by Jane Doe
filter_results = vectorstore.similarity_search(
    "What is supervised learning?",
    k=2,
    filter={"author": "Jane Doe"}
)

print("Filtered retrieval results:")
for r in filter_results:
    print(f"  [{r.metadata['chunk_id']}] {r.page_content[:80].strip()}")
    print(f"  Author: {r.metadata['author']} | Version: {r.metadata['version']}\n")
